# CodeOrbit real-agent test - Kaggle (unattended, two GPUs)

A local model localises each SWE-bench issue twice: with the usual agent tools
(grep, read_file, list_dir), and with those plus CodeOrbit's MCP tools. One
model, two GPUs: an Ollama server per GPU, each working through half the tasks
(`--shard 1/2`, `--shard 2/2`); the report pools them into one paired test.

### Once, in the right-hand panel (Settings)
- **Accelerator:** GPU T4 x2
- **Internet:** On

### To run
1. Check the settings cell below.
2. **Save Version -> Save & Run All (Commit) -> Save.** You can close the browser.
3. Later: open the notebook -> latest version -> **Output** -> download
   `codeorbit-agent-runs.zip`.

### If it stops early
Everything stops cleanly within `TOTAL_HOURS` and is saved. To continue: **Add
Input -> Your Work -> this notebook**, set `RESUME = 'auto'` and
`CHECK_FIRST = False`, and Save & Run All again. Each job finds its own
unfinished run folder; finished issues are skipped.

## Settings

In [ ]:
MODEL = 'gpt-oss:20b'   # 'qwen3:14b' for the earlier runs
SPLITS = ['multilingual', 'verified']   # run one after the other, in this order
                   # 'multilingual' - Go, Java, Rust, PHP, Ruby, C/C++ issues
                   # 'verified'     - Python issues; in 'main2' and 'dev' mode mixed
                   #                  with JavaScript/TypeScript ones (from Multilingual)
                   # 'lite'         - the earlier qwen3:8b test's issues
MODE = 'main2'     # 'main2' - the second test: 100 + 100 fresh issues (bench/PREREGISTRATION.md)
                   # 'main'  - the first test's issues (already run)
                   # 'dev'   - 30 + 30 issues for trying a change
                   # 'pilot' - 3 issues outside each main sample: checks the pipeline
PROFILE = 'full'   # how CodeOrbit reaches the model in the with condition:
                   # 'full'   - its tools (the pre-registered tests)
                   # 'inject' - no tools; its ranked code is added to the issue
CHECK_FIRST = True # first run 7 pilot issues (Python, Go, Java, Rust, JavaScript) and
                   # check them: every session ran, CodeOrbit's tools were called,
                   # answers came back. The main run starts only if that passes.
TOTAL_HOURS = 11.5 # for ALL splits together: everything stops cleanly and is saved
                   # before this; Kaggle stops any session at 12 h
RESUME = {}        # to continue a stopped run: add this notebook's earlier version as
                   # input (Add Input -> Your Work) and set RESUME = 'auto' - each job
                   # picks up its own unfinished folder. (Or a dict as before.)

## 1. GPUs, get CodeOrbit

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv

RUNS = '/kaggle/working/codeorbit-agent-runs'
SRC = '/tmp/codeorbit'
os.makedirs(RUNS, exist_ok=True)

%cd /tmp
!rm -rf {SRC}
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git {SRC}
%cd {SRC}
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 2. Ollama, one server per GPU

`ollama ps` for each should list the model at 100% GPU.

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
SERVERS = {1: ('0', 11434), 2: ('1', 11435)}    # shard -> (GPU, port)
for shard, (gpu, port) in SERVERS.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, OLLAMA_HOST=f'127.0.0.1:{port}',
               OLLAMA_MAX_LOADED_MODELS='2', OLLAMA_NUM_PARALLEL='1', OLLAMA_KEEP_ALIVE='-1')
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(f'/tmp/ollama{shard}.log', 'w'),
                     stderr=subprocess.STDOUT)
for _, port in SERVERS.values():
    for _ in range(60):
        try:
            requests.get(f'http://127.0.0.1:{port}/api/version', timeout=2); break
        except Exception:
            time.sleep(1)
!OLLAMA_HOST=127.0.0.1:11434 ollama pull nomic-embed-text
!OLLAMA_HOST=127.0.0.1:11434 ollama pull {MODEL}
for _, port in SERVERS.values():     # pulled once to disk; load it on each GPU
    requests.post(f'http://127.0.0.1:{port}/api/chat', timeout=600, json={
        'model': MODEL, 'messages': [{'role': 'user', 'content': 'hi'}], 'stream': False,
        'options': {'num_ctx': 16384}}).raise_for_status()
!OLLAMA_HOST=127.0.0.1:11434 ollama ps
!OLLAMA_HOST=127.0.0.1:11435 ollama ps

## 3. Download SWE-bench (Lite is needed to exclude it from Verified)

In [ ]:
for split in ['lite', 'verified', 'multilingual']:
    !python scripts/swebench.py download --split {split}

## 4. Run both halves

Each half runs in its own process, on its own GPU, with its own copies of the
repositories. This cell prints progress every 10 minutes until both finish.

In [ ]:
import re, shutil
# issue selection is fixed in bench/PREREGISTRATION.md (verified) and bench/README.md (lite)
selection = {
    ('verified', 'pilot'): '--split verified --instances '
                           'psf__requests-1921,pytest-dev__pytest-7205,pytest-dev__pytest-10081',
    ('verified', 'main'):  '--split verified --exclude-split lite --sample 100 --seed 3',
    ('multilingual', 'pilot'): '--split multilingual --instances '
                               'gin-gonic__gin-3741,google__gson-2061,sharkdp__bat-1892',
    ('multilingual', 'main'):  '--split multilingual --sample 100 --seed 3',
    ('multilingual', 'dev'):   '--split multilingual --exclude-sample 100 --exclude-seed 3 --sample 30 --seed 5',
    # dev 'verified': 15 Python issues and 15 JavaScript/TypeScript ones (Verified is
    # Python-only), none in any main sample or in the Multilingual dev set
    # the second test (bench/PREREGISTRATION.md): 100 + 100 issues no earlier run touched
    ('multilingual', 'main2'): '--split multilingual --exclude-sample 100 --exclude-seed 3 '
                               '--langs go,java,rust,php,ruby,c,cpp --sample 100 --seed 11',
    ('verified', 'main2'): [
        ('verified-python', '--split verified --exclude-split lite --exclude-sample 100 '
                            '--exclude-seed 3 --sample 72 --seed 11'),
        ('verified-jsts', '--split multilingual --exclude-sample 100 --exclude-seed 3 '
                          '--langs javascript,typescript')],
    ('verified', 'dev'): [
        ('verified-python', '--split verified --exclude-split lite --exclude-sample 100 '
                            '--exclude-seed 3 --sample 15 --seed 5'),
        ('verified-jsts', '--split multilingual --exclude-sample 100 --exclude-seed 3 '
                          '--exclude-next 30:5 --langs javascript,typescript --sample 15 --seed 7')],
    ('lite', 'pilot'):     '--split lite --exclude-sample 60 --exclude-seed 0 --sample 10 --seed 2',
    ('lite', 'main'):      '--split lite --exclude-sample 60 --exclude-seed 0 --sample 100 --seed 1',
}

# The check: pilot issues outside every main sample (one JavaScript issue from the
# first test's sample, since no JS/TS issue is left outside both), run first.
CHECK_JOBS = [
    ('check-python', '--split verified --instances '
                     'psf__requests-1921,pytest-dev__pytest-7205,pytest-dev__pytest-10081'),
    ('check-multilingual', '--split multilingual --instances '
                           'gin-gonic__gin-3741,google__gson-2061,sharkdp__bat-1892,axios__axios-4731'),
]

# One time budget for everything after this point. Each split gets what is left;
# the harness stops between issues when its --max-hours is up and saves every
# finished issue, so nothing is lost if the budget runs out mid-split.
deadline = time.time() + TOTAL_HOURS * 3600 - 15 * 60   # 15 min kept for reports + zip
logs = []


def earlier_run(sel, shard):
    """With RESUME = 'auto': this job's run folder from an earlier version of this
    notebook added as input - same model, profile, split, languages and sample."""
    want = dict(re.findall(r'--([a-z-]+) (\S+)', sel))
    for meta_path in sorted(glob.glob('/kaggle/input/**/meta.json', recursive=True)):
        run = os.path.dirname(meta_path)
        if f'shard{shard}of2' not in os.path.basename(run) or '/check/' in run:
            continue
        m = json.load(open(meta_path))
        s, r = m['dataset'].get('selection', {}), m['run']
        if (r.get('model') == MODEL and r.get('profile', 'full') == PROFILE
                and m['dataset']['name'].endswith(want.get('split', ''))
                and s.get('langs') == want.get('langs')
                and str(s.get('sample')) == str(want.get('sample', None))
                and str(s.get('seed')) == str(want.get('seed', 0))):
            return run
    return None


def run_jobs(jobs, out, label, cap_h=None):
  for split, sel in jobs:
    left_h = (deadline - time.time()) / 3600
    if cap_h:
        left_h = min(left_h, cap_h)
    if left_h < 0.25:
        print(f'no time left for {split}; resume it later with RESUME')
        continue
    print(f'\n######## {split} ({label}): up to {left_h:.1f} h')
    os.makedirs(out, exist_ok=True)
    procs = {}
    for shard, (gpu, port) in SERVERS.items():
        tag = f'{split}-shard{shard}'
        src = RESUME.get((split, shard)) if isinstance(RESUME, dict) else (
            earlier_run(sel, shard) if RESUME == 'auto' else None)
        if src:
            print(f'resuming {src}')
            dst = os.path.join(RUNS, os.path.basename(src.rstrip('/')))
            if not os.path.exists(dst):
                shutil.copytree(src, dst)            # /kaggle/input is read-only
            cmd = f'python scripts/agent_eval.py run --resume {dst}'
        else:
            cmd = (f'python scripts/agent_eval.py run --model {MODEL} {sel} --shard {shard}/2'
                   f' --profile {PROFILE}')
        cmd += (f' --embed --ollama http://127.0.0.1:{port} --checkouts /tmp/checkouts-shard{shard}'
                f' --out {out} --max-hours {left_h:.2f}')
        print(cmd)
        procs[tag] = (subprocess.Popen(cmd, shell=True, stdout=open(f'/tmp/{tag}.log', 'w'),
                                       stderr=subprocess.STDOUT), f'/tmp/{tag}.log')
        logs.append(f'/tmp/{tag}.log')
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(600)
        for m, (p, log) in procs.items():
            lines = open(log, encoding='utf-8', errors='replace').read().splitlines()
            tasks = [l for l in lines if l.startswith('[')]
            print(f'{time.strftime("%H:%M")} {m}: {tasks[-1] if tasks else "starting"}'
                  f'{"  (finished)" if p.poll() is not None else ""}', flush=True)
    for m, (p, log) in procs.items():
        print(f'\n===== {m} (exit {p.returncode}) - last lines of its log')
        print('\n'.join(open(log, encoding='utf-8', errors='replace').read().splitlines()[-25:]))


def check_ok(out):
    """Did the pilot work end to end? Problems are printed, one per line."""
    problems, rows = [], []
    for run in glob.glob(f'{out}/*-agent-*'):
        meta = json.load(open(f'{run}/meta.json'))
        got = [json.loads(l) for l in open(f'{run}/runs.jsonl') if l.strip()]
        rows += got
        if len(got) < 2 * meta['dataset'].get('tasks', 0):
            problems.append(f'{os.path.basename(run)}: {len(got)} sessions for '
                            f'{meta["dataset"].get("tasks")} issues')
    if not rows:
        return ['no session was recorded']
    for cond in ('without', 'with'):
        r = [x for x in rows if x['condition'] == cond]
        answered = sum(bool(x.get('answer')) for x in r)
        print(f'check {cond}: {len(r)} sessions, {answered} answered, '
              f'{sum(x.get("fn_hit", False) for x in r)} found the function')
        if r and answered < len(r) / 2:
            problems.append(f'{cond}: only {answered} of {len(r)} sessions answered')
    co = sum(n for x in rows if x['condition'] == 'with'
             for t, n in x.get('tool_calls', {}).items() if t.startswith('codeorbit'))
    print(f'check: CodeOrbit tool calls in the with sessions: {co}')
    if not co:
        problems.append('no with-CodeOrbit session called a CodeOrbit tool')
    return problems


import glob, json
CHECK_OUT = '/kaggle/working/check-runs'
go = True
if CHECK_FIRST:
    run_jobs(CHECK_JOBS, CHECK_OUT, 'check', cap_h=1.5)
    problems = check_ok(CHECK_OUT)
    for p in problems:
        print('CHECK FAILED:', p)
    go = not problems
    print('\ncheck passed - starting the main run' if go else
          '\ncheck failed - the main run is skipped; see above')
if go:
    jobs = []                   # (name, selection): a split, or each part of one
    for split in SPLITS:
        sel = selection[(split, MODE)]
        jobs += sel if isinstance(sel, list) else [(split, sel)]
    run_jobs(jobs, RUNS, MODE)

## 5. Results and download

In [ ]:
import glob, json
# Pooled per sample: the two halves of one split together, never across splits.
by_split = {}
for run in sorted(glob.glob(f'{RUNS}/*-agent-*')):
    d = json.load(open(f'{run}/meta.json'))['dataset']
    sel = d.get('selection') or {}
    name = d['name'] + (f" ({sel['langs']})" if sel.get('langs') else '')
    by_split.setdefault(name, []).append(run)
jsts = [v for k, v in by_split.items() if 'javascript' in k]
if MODE in ('dev', 'main2') and 'swebench-verified' in by_split and jsts:
    # the mixed dev set, Python + JavaScript/TypeScript, also reported together
    by_split['mixed: python + javascript/typescript'] = by_split['swebench-verified'] + jsts[0]
for name, runs in by_split.items():
    print(f'\n################ {name}')
    !python scripts/agent_eval.py report --pool {' '.join(runs)}
for log in logs:
    if os.path.exists(log):
        shutil.copy(log, RUNS)                   # keep the logs with the results
if os.path.isdir(CHECK_OUT):                     # the check's runs go in the zip too
    shutil.copytree(CHECK_OUT, f'{RUNS}/check', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/codeorbit-agent-runs', 'zip', RUNS)
print('download /kaggle/working/codeorbit-agent-runs.zip from the Output tab')